In [1]:
# Combine scraped data without detailed location and description to maukerja_compiled.csv
import pandas as pd
import glob
import re
import os
import shutil

os.makedirs("raw_data_processed", exist_ok=True)

new_files = glob.glob("raw_data/maukerja_*.csv")

if not new_files:
    print("No new files in raw_data.")
else:
    # Load new files
    df_new = pd.concat([pd.read_csv(f) for f in new_files], ignore_index=True)
    print(f"New records loaded: {len(df_new):,}")

    # Load existing compiled if it exists
    compiled_path = "maukerja_compiled.csv"
    if os.path.exists(compiled_path):
        df_existing = pd.read_csv(compiled_path, dtype={'job_id': str})
        print(f"Existing compiled records: {len(df_existing):,}")
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new

    print(f"Total before dedup: {len(df_combined):,}")

    # Extract and normalize job_id
    def extract_job_id(row):
        job_id = str(row['job_id']).strip() if pd.notna(row['job_id']) else ''
        url = str(row['url']).strip() if pd.notna(row['url']) else ''
        if job_id in ('', 'nan') and url not in ('', 'nan'):
            match = re.search(r'jobId=(\d+)', url)
            return match.group(1) if match else 'N/A'
        return job_id if job_id not in ('', 'nan') else 'N/A'

    df_combined['job_id'] = df_combined.apply(extract_job_id, axis=1)
    df_combined['job_id'] = (
        df_combined['job_id']
        .astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.replace(r'\s+', '', regex=True)
    )

    # Dedup
    df_final = df_combined[df_combined['job_id'] != 'N/A'].drop_duplicates(subset=['job_id'], keep='first')
    print(f"Total unique jobs: {len(df_final):,}")

    # Save
    df_final.to_csv(compiled_path, index=False, encoding='utf-8-sig')
    print(f"Saved to {compiled_path}")

    # Move processed files
    for f in new_files:
        filename = os.path.basename(f)
        dest = os.path.join("raw_data_processed", filename)
        shutil.move(f, dest)
        print(f"  Moved {filename}")

New records loaded: 8,264
Existing compiled records: 104,029
Total before dedup: 112,293
Total unique jobs: 110,300
Saved to maukerja_compiled.csv
  Moved maukerja_2026-05-16_23-52-06.csv
  Moved maukerja_2026-05-17_16-15-01.csv


In [ ]:
# Separate files for task delegation
# Futher separate to run on different notebooks at the same time
import pandas as pd
import os

# Config
input_file = "maukerja_compiled.csv"
output_file_prefix = "compiled_no_desc"

df = pd.read_csv(input_file)

# Force string dtype
for col in ["scraped_location", "scraped_description"]:
    if col not in df.columns:
        df[col] = pd.array([""] * len(df), dtype="string")
    df[col] = df[col].astype("string").fillna("")

# Split into has description and no description
has_desc = df[
    (df["scraped_description"] != "") &
    (df["scraped_description"] != "Job Closed") &
    (df["scraped_description"] != "Error")
].copy()

no_desc = df[
    (df["scraped_description"] == "") |
    (df["scraped_description"] == "Job Closed") |
    (df["scraped_description"] == "Error")
].copy()

print(f"Has description: {len(has_desc):,}")
print(f"No description:  {len(no_desc):,}")

# Save the has_description file (no need to split, already done)
has_desc.to_csv("compiled_has_desc.csv", index=False, encoding="utf-8-sig")
print(f"Saved compiled_has_desc.csv")

# Split no_desc into 3 equal chunks
chunk_size = len(no_desc) // 3
chunks = [
    no_desc.iloc[:chunk_size],
    no_desc.iloc[chunk_size:chunk_size*2],
    no_desc.iloc[chunk_size*2:]
]

for i, chunk in enumerate(chunks):
    path = f"{output_file_prefix}{i}.csv"
    chunk.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"File {i}: {len(chunk):,} rows → {path}")

C:\Users\yuyla\AppData\Local\Temp\ipykernel_11816\1570932511.py:10: DtypeWarning: Columns (0: job_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


Has description: 104,055
No description:  6,245
Saved compiled_has_desc.csv
File 0: 2,081 rows → compiled_no_desc0.csv
File 1: 2,081 rows → compiled_no_desc1.csv
File 2: 2,083 rows → compiled_no_desc2.csv


In [4]:
# Scrap Again For Description
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random

FILENAME = "compiled_no_desc0.csv"      # Change file names accordingly
df = pd.read_csv(FILENAME)   

def scrape_job_detail(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9",
        }
        resp = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Lokasi Kerja
        location_div = soup.find("div", id="job-locations")
        if location_div:
            items = location_div.find_all("li")
            location = "; ".join(li.get_text(strip=True) for li in items) if items else location_div.get_text(strip=True)
        else:
            location = ""

        # Penerangan Kerja
        desc_div = soup.find("div", id="job-description")
        if desc_div:
            description = desc_div.get_text(separator="\n", strip=True)
        else:
            description = ""
        
        if not location and not description:
            return "Job Closed", "Job Closed"

        return location, description

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return "Error", "Error"

# Force string dtype from the start
if "scraped_location" not in df.columns:
    df["scraped_location"] = pd.array([""] * len(df), dtype="string")
if "scraped_description" not in df.columns:
    df["scraped_description"] = pd.array([""] * len(df), dtype="string")

# Also cast if columns already exist but are float
df["scraped_location"] = df["scraped_location"].astype("string").fillna("")
df["scraped_description"] = df["scraped_description"].astype("string").fillna("")

to_scrape = df[
    (df["scraped_location"].fillna("") == "") 
   |(df["scraped_location"] == "Job Closed")   # Recheck for Description set as Job Closed
].index

print(f"Scraping {len(to_scrape)} jobs...")

for i, idx in enumerate(to_scrape):
    url = df.at[idx, "url"]
    if pd.isna(url) or url in ("", "N/A", "nan"):
        continue

    loc, desc = scrape_job_detail(url)
    df.loc[idx, "scraped_location"] = str(loc)
    df.loc[idx, "scraped_description"] = str(desc)

    # Save every 50 rows in case it crashes
    if i % 50 == 0:
        df.to_csv(FILENAME, index=False, encoding="utf-8-sig")
        print(f"Progress: {i}/{len(to_scrape)} — last: {url}")

    # Random delay to avoid getting blocked
    time.sleep(random.uniform(1.5, 3.5))

df.to_csv(FILENAME, index=False, encoding="utf-8-sig")
print("Done!")

Scraping 3576 jobs...
Progress: 0/3576 — last: https://www.maukerja.my/job/26135007-account-executive-costing
Progress: 50/3576 — last: https://www.maukerja.my/job/26031005-assistant-secretary-personal
Progress: 100/3576 — last: https://www.maukerja.my/job?jobId=26097946&slug=safety-health-officer-lss-4-months-contract
Progress: 150/3576 — last: https://www.maukerja.my/job?jobId=26126978&slug=admin-officer
Progress: 200/3576 — last: https://www.maukerja.my/job/25948816-rakan-cukai
Progress: 250/3576 — last: https://www.maukerja.my/job?jobId=26112216&slug=sales-person-cum-admin
Progress: 300/3576 — last: https://www.maukerja.my/job?jobId=26076325&slug=electrical-section-head
Progress: 350/3576 — last: https://www.maukerja.my/job/25907222-delivery-assistant
Progress: 400/3576 — last: https://www.maukerja.my/job/25870346-operation-supervisor-manager
Progress: 450/3576 — last: https://www.maukerja.my/job?jobId=26074803&slug=senior-software-engineer
Progress: 500/3576 — last: https://www.ma

In [5]:
# merge scraped.py
import pandas as pd
import glob

scraped_files = glob.glob("compiled_*.csv")
df_scraped = pd.concat([pd.read_csv(f) for f in scraped_files], ignore_index=True)
df_scraped.drop_duplicates(subset=['job_id'], keep='first')
print(len(df_scraped))
df_scraped.to_csv("maukerja_compiled.csv", index=False, encoding="utf-8-sig")

C:\Users\yuyla\AppData\Local\Temp\ipykernel_11816\1460965271.py:6: DtypeWarning: Columns (0: job_id, 1: experience_level) have mixed types. Specify dtype option on import or set low_memory=False.
  df_scraped = pd.concat([pd.read_csv(f) for f in scraped_files], ignore_index=True)


110300


In [6]:
# CSV TO GZIP
import pandas as pd
df = pd.read_csv("maukerja_compiled.csv")
df.to_csv("maukerja_compiled.gz", index=False, compression="gzip")

C:\Users\yuyla\AppData\Local\Temp\ipykernel_11816\1204288235.py:3: DtypeWarning: Columns (0: job_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("maukerja_compiled.csv")


In [ ]:
print(f"\n{'=' * 50}")
print(f"COMBINED JOBS SUMMARY")
print(f"{'=' * 50}")
print(f"Total unique jobs:        {len(df_combined):,}")
# print(f"Total CSV files merged:   {len(all_files)}")

print(f"\n--- Top 10 Locations ---")
print(df_combined['location'].value_counts().head(10).to_string())

print(f"\n--- Job Type Breakdown ---")
print(df_combined['job_type'].value_counts().to_string())

print(f"\n--- Top 10 Companies (most listings) ---")
print(df_combined['company'].value_counts().head(10).to_string())

print(f"\n--- Top 10 Most Common Job Titles ---")
print(df_combined['title'].value_counts().head(10).to_string())

print(f"\n--- Skills (Top 15) ---")
# Skills are comma-separated, need to split and count individually
all_skills = df_combined['skills'].dropna()
all_skills = all_skills[all_skills != 'N/A']
skill_counts = (
    all_skills
    .str.split(', ')
    .explode()
    .str.strip()
    .value_counts()
    .head(15)
)
print(skill_counts.to_string())

print(f"\n--- Date Posted Breakdown ---")
print(df_combined['date_posted'].value_counts().head(10).to_string())

print(f"\n--- Missing Data ---")
for col in df_combined.columns:
    missing = (df_combined[col] == 'N/A').sum()
    if missing > 0:
        pct = (missing / len(df_combined)) * 100
        print(f"  {col}: {missing:,} missing ({pct:.1f}%)")